In [7]:
%pip install pandas numpy scikit-learn joblib


   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 4.0 MB/s eta 0:00:03
   ---------- ----------------------------- 2.6/9.9 MB 7.5 MB/s eta 0:00:01
   --------------- ------------------------ 3.9/9.9 MB 6.8 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.9 MB 7.6 MB/s eta 0:00:01
   -------------------------------- ------- 8.1/9.9 MB 8.3 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.9 MB 8.0 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 7.7 MB/s  0:00:01
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.6 MB 7.8 MB/s eta 0:00:02
   ---------- ----------------------------- 3.1/12.6 MB 7.9 MB/s eta 0:00:02
   --------------- ------------------------ 5.0/12.6 MB 8.1 MB/s eta 0:00:01
   -------------------- ------------------- 6.3/12.6 MB 8.2 MB/s eta 0:00:01
   ---------------------

Tema: "Predicción de Supervivencia y Costo de Boletos en el Titanic: Un enfoque integrador con Machine Learning, Streamlit y Realidad Aumentada"

El Modelo de Regresión Logística (Clasificación): Predecirá si un pasajero sobrevive (Survived: 0 = No, 1 = Sí) según variables como su edad, género y clase.

El Modelo de Regresión Lineal (Predicción): Predecirá el precio exacto del boleto (Fare) que pagó un pasajero en función de su clase, puerto de embarque y si viajaba con familia.

El objetivo es entender qué factores influyeron en la supervivencia del naufragio del Titanic y predecir los costos de las tarifas para entender la estructura de precios de la época.

In [17]:
import pandas as pd
import numpy as np

# Cargar el dataset
df = pd.read_csv('titanic.csv')

# Ver las primeras filas y estructura
print(df.head())
print(df.info())
# 1. Rellenar edades faltantes con la mediana
df['Age'] = df['Age'].fillna(df['Age'].median())

# 2. Rellenar puertos de embarque faltantes con el más común
df['Embarked'] = df['Embarked'].fillna('S')

# 3. Convertir la columna 'Sex' a números (0 = hombre, 1 = mujer)
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# 4. Convertir 'Embarked' a números usando variables dummies
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

# Seleccionar solo las columnas numéricas finales que usaremos
columnas_finales = ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
df_limpio = df[columnas_finales].astype(float)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
import joblib # Para guardar los modelos

# --- MODELO 1: Regresión Logística (¿Sobrevivió o no?) ---
X_log = df_limpio[['Pclass', 'Sex', 'Age', 'Fare']]
y_log = df_limpio['Survived']

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_log, y_log, test_size=0.2, random_state=42)

modelo_logistica = LogisticRegression()
modelo_logistica.fit(X_train_l, y_train_l)

# --- MODELO 2: Regresión Lineal (Predicción del Precio 'Fare') ---
X_lin = df_limpio[['Pclass', 'Sex', 'Age', 'Survived']]
y_lin = df_limpio['Fare']

X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(X_lin, y_lin, test_size=0.2, random_state=42)

modelo_lineal = LinearRegression()
modelo_lineal.fit(X_train_lin, y_train_lin)

# Guardar modelos para usarlos en Streamlit más tarde
joblib.dump(modelo_logistica, 'modelo_logistica.pkl')
joblib.dump(modelo_lineal, 'modelo_lineal.pkl')
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

# Evaluar Logística
pred_log = modelo_logistica.predict(X_test_l)
print("Precisión Regresión Logística:", accuracy_score(y_test_l, pred_log))

# Evaluar Lineal
pred_lin = modelo_lineal.predict(X_test_lin)
print("Error Cuadrático Medio Lineal:", mean_squared_error(y_test_lin, pred_lin))
print("R2 Score (Varianza explicada):", r2_score(y_test_lin, pred_lin))


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c